In [0]:
from pyspark.sql.functions import DataFrame
from pyspark.sql import functions as F
import pandas as pd

In [0]:
df_renewal: DataFrame = spark.table("workspace.`ds-raw-datasets`.raw_renewal_calls")
df_renewal.display()

In [0]:
pd_renewal: pd.DataFrame = df_renewal.toPandas()
pd_renewal.columns

In [0]:
ren_pd = pd_renewal.dropna(subset=['co_ref', "call_date"])
ren_pd.groupby(['co_ref']).agg({'call_date': 'max'}, columns=['call_date']).sort_values(by='call_date', ascending=False).display()
# print(ren_pd.shape, pd_renewal.shape)

In [0]:
%sql
use workspace.`ds-raw-datasets`;

with renewal_calls as (
  select co_ref, max(call_date) as call_date from raw_renewal_calls group by co_ref
) select * from f_billings b join renewal_calls r on b.co_ref = r.co_ref and r.call_date between b.prospect_renewal_date and b.closed_date order by b.co_ref, r.call_date

In [0]:
df_renewal.select('customer_renewal_response_category').distinct().show()

In [0]:
%sql
use workspace.`ds-raw-datasets`;
select co_ref, call_date, customer_renewal_response_category from raw_renewal_calls r where co_ref is not null group by co_ref, call_date, customer_renewal_response_category having customer_renewal_response_category is not null and customer_renewal_response_category not like '%Not Mentioned%' order by co_ref, call_date;

In [0]:
df_cc: DataFrame = spark.table("workspace.`ds-raw-datasets`.raw_cc_calls")
df_cc.display()